In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
# simple dataframe w/ automatic column headers
df = session.create_dataframe([[1, 2, 3, 4], [5, 6, 7, 8]])
df

In [ ]:
# same dataframe w/ named column headers (and generated SQL query)
df = session.create_dataframe(
    [[1, 2, 3, 4], [5, 6, 7, 8]],
    schema=["a", "b", "c", "d"])
df

df = session.create_dataframe(
    [[1, 2, 3, 4], [5, 6, 7, 8]]
    ).to_df("a", "b", "c", "d")
df

In [ ]:
df.schema
df.columns
df.queries
res = df.describe()
res

In [ ]:
# renders dataframe as text (max 10 rows by def)
df.show()

In [ ]:
# renders dataframe as collection of Row objects
df.collect()

In [ ]:
# extract data from Row
df.collect()[0].as_dict()

In [ ]:
print(f'Total rows: {df.count()}')
print(f'Values 1 in first row: {df.collect()[0].count(1)}')

In [ ]:
from snowflake.snowpark.functions import col

# select columns
df.select("c", col("c"), df["c"], df.c, df.col("c"))

In [ ]:
# column aliases
df2 = df.select("c", col("c").alias("c2"), df["c"].as_("c3"), df.c.name("c4"))
df2.queries["queries"][0]
df2

In [ ]:
from snowflake.snowpark.types import StringType

# cast values to another data types (string)
df.select(
    df.c.cast(StringType()).alias("c1"),
    df.c.astype(StringType()).alias("c2"))

In [ ]:
from snowflake.snowpark.functions import sql_expr

# SQL expression(s)
df2 = df.select_expr("a+2", "cast(b as string)")
df2

df2 = df.select(
    sql_expr("a+2").as_("ex1"),
    sql_expr("cast(b as string)").as_("ex2"))
df2

In [ ]:
# remove existing columns
df.drop("b", "d")

In [ ]:
# rename columns
df.rename({col("a"): "a2", "b": "b2"})

In [ ]:
from snowflake.snowpark.functions import lit

# add new columns
df.with_column("e", df.a + df.b
    ).with_columns(["f", "g"], [lit("11"), lit("12")]
    ).with_column_renamed("a", "a2")

In [ ]:
# ~df2.collect(block=False) --> AsyncJob obj
job = df2.collect_nowait()
job.query
df = job.result()
df
df = job.to_df()
df
job.is_done()

In [ ]:
# (try to) cancel AsyncJob
job = session.sql("select SYSTEM$WAIT(3)").collect_nowait()
job.cancel()
job.is_done()

In [ ]:
job.is_done()